# Gıda Üretim Şirketinde Çoklu Ürün Talep Tahmini ve Envanter Optimizasyonu

**Veri Bilimi & Makine Öğrenmesi Bitirme Projesi**

Hazırlayan: Emre Çimen
Danışman: Hakan Çelik

---


## Bölüm 1: Problem Tanımı ve Veri Toplama

### 1.1 İş Problemi

Konserve gıda üretimi yapan bir şirkette (üretim planlama stajımda gözlemlediğim
Oracle ERP tabanlı süreçlere benzer şekilde), **malzeme planlama ve üretim
çizelgeleme kararları büyük ölçüde talep tahminine dayanır.** Talebin
olduğundan düşük tahmin edilmesi **stoksuz kalmaya (stockout)** ve üretim
hattında acil/plansız çizelgeleme değişikliklerine; olduğundan yüksek tahmin
edilmesi ise **fazla stok, depolama maliyeti ve son kullanma tarihi riskine**
yol açar.

**Proje hedefi:** Şirketin 5 farklı ürününün (SKU) önümüzdeki dönemdeki günlük
satış/talep miktarını, geçmiş satış, fiyat, promosyon ve takvim (tatil/mevsim)
verilerini kullanarak tahmin etmek; bu tahminlere dayanarak her ürün için
**emniyet stoğu ve yeniden sipariş noktası** önerisi sunmak.

Somutlaştırılmış hedef: *"Her ürünün günlük talebini ortalama %15'in altında
hata (MAPE) ile tahmin ederek, planlama ekibinin stoksuz kalma riskini önceden
görmesini sağlamak."*

### 1.2 Veri Kaynağı

Gerçek bir üretim/ERP sisteminden (örn. Dardanel'in Oracle ERP'si) veri
çekmek, şirket gizliliği ve erişim kısıtları nedeniyle bu bitirme projesi
kapsamında mümkün değildir. Bunun yerine, **stajda gözlemlediğim gerçek
süreç dinamiklerini (haftalık/yıllık mevsimsellik, promosyon etkisi, resmi
tatillerde üretim/sevkiyat düşüşü, veri kalitesi sorunları) yansıtan kontrollü
bir sentetik veri seti ürettim** (`src/generate_data.py`). Bu veri seti:

- 5 ürün (SKU) x 3 yıl (2022-2024) = **5.480 satır günlük gözlem**,
- Trend, haftalık ve yıllık mevsimsellik (yaz ayları + Ramazan öncesi
  stoklama etkisi), promosyon ve resmi tatil etkisi,
- **Kasıtlı olarak eklenmiş** %3 oranında eksik satış verisi ve %1.5
  oranında aykırı değer (gerçek ERP verilerinde sıkça karşılaşılan veri
  kalitesi sorunlarını simüle etmek için)

içerir. **Kısıt:** Bu veri gerçek satış rakamlarını yansıtmadığından, projenin
amacı gerçek bir tahmin doğruluğu iddia etmek değil, **uçtan uca bir talep
tahmini metodolojisini doğru şekilde uygulamaktır.** Bu kısıt Bölüm 6'da
tekrar ele alınmıştır.

### 1.3 Başarı Kriteri

| Metrik Türü | Metrik | Hedef |
|---|---|---|
| Teknik | MAPE (Mean Absolute Percentage Error) | < %15 |
| Teknik | RMSE | Baseline (naive forecast) modelden düşük |
| İş | Stoksuz kalma riski taşıyan ürün-gün sayısı | Mevcut duruma göre azalma |
| İş | Planlama ekibinin karar süresi | Tahmine dayalı otomatik uyarı ile kısalma |


## Bölüm 2: Veri Hazırlama ve Temizleme

Önce veri setini yükleyip genel yapısını inceleyelim.

In [ ]:
import os
from pathlib import Path

# Matplotlib önbelleğinin yazılabilir bir klasörde tutulması, notebook'un
# farklı bilgisayarlarda sorunsuz çalışmasına yardımcı olur.
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="deep")


def dosyayi_bul(dosya_adi):
    """Notebook kökten, upload klasöründen veya data klasöründen çalışabilir."""
    cwd = Path.cwd()
    adaylar = [
        cwd / dosya_adi,
        cwd / "upload" / dosya_adi,
        cwd / "data" / dosya_adi,
        cwd.parent / "data" / dosya_adi,
    ]
    for aday in adaylar:
        if aday.exists():
            return aday.resolve()
    raise FileNotFoundError(
        f"{dosya_adi} bulunamadı. CSV dosyasını notebook ile aynı klasöre "
        "veya data/ klasörüne koyun."
    )


DATA_PATH = dosyayi_bul("talep_verisi_ham.csv")
WORK_DIR = DATA_PATH.parent
OUTPUT_DIR = WORK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=["tarih"])
df = df.sort_values(["urun_kodu", "tarih"]).reset_index(drop=True)

print("Kullanılan veri dosyası:", DATA_PATH)
print("Veri boyutu:", df.shape)
print("\nSütun tipleri:")
print(df.dtypes)
df.head()


In [ ]:
# Temel istatistikler
df.describe(include="all").T

### 2.1 Eksik Veri Stratejisi

Önce eksik değerlerin oranına bakalım.

In [ ]:
eksik_oran = (df.isna().mean() * 100).round(2)
print(eksik_oran[eksik_oran > 0])

fig, ax = plt.subplots(figsize=(6, 3))
sns.barplot(
    x=eksik_oran[eksik_oran > 0].index,
    y=eksik_oran[eksik_oran > 0].values,
    ax=ax,
)
ax.set_ylabel("Eksik Değer Oranı (%)")
ax.set_title("Sütun Bazında Eksik Veri Oranı")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eksik_veri_orani.png", dpi=140, bbox_inches="tight")
plt.show()


**Gözlem:** `satis_adedi` sütununda %2.99, `birim_fiyat` sütununda %0.49
oranında eksik değer var. Bu seviyeler (<%5), satır silmeyi gerektirecek kadar
yüksek değil ancak veriyi tamamen atlamak da veri kaybına (özellikle zaman
serisi sürekliliğini bozacağı için) yol açar.

**Neden ortalama/medyan ile doldurmadım:** Satış verisi güçlü bir zamansal
yapıya (trend + mevsimsellik) sahiptir. Genel ortalama ile doldurmak, örneğin
yaz ayındaki bir eksik günü kışın ortalama değeriyle doldurup yapay bir
düşüş/artış yaratabilir.

**Seçilen yöntem:** Her ürün için ayrı ayrı, **zaman sıralı doğrusal
interpolasyon** (`interpolate(method="linear")`). Bu yöntem, eksik günün
komşu günlerindeki gerçek trend ve mevsimsel seviyeyi koruyarak daha
gerçekçi bir tahmin sağlar. Fiyat verisi için de aynı mantık geçerli (fiyatlar
günden güne küçük değişimlerle hareket eder, ani sıçrama beklenmez).

In [ ]:
df["satis_adedi"] = df.groupby("urun_kodu")["satis_adedi"].transform(
    lambda s: s.interpolate(method="linear", limit_direction="both")
)
df["birim_fiyat"] = df.groupby("urun_kodu")["birim_fiyat"].transform(
    lambda s: s.interpolate(method="linear", limit_direction="both")
)

print("Interpolasyon sonrası toplam eksik değer:", df.isna().sum().sum())

### 2.2 Aykırı Değer (Outlier) Yönetimi

Aykırı değerleri **ürün bazında** tespit ediyoruz çünkü ürünlerin satış
ölçekleri birbirinden çok farklı (örn. Ton Balığı 80g günde ~850 adet
satarken, Somon Konservesi ~180 adet satıyor); tüm veri setine tek bir IQR
sınırı uygulamak küçük hacimli ürünlerin tüm gözlemlerini yanlışlıkla aykırı
işaretler.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=False)
for ax, (sku, grup) in zip(axes, df.groupby("urun_kodu")):
    sns.boxplot(y=grup["satis_adedi"], ax=ax, color="skyblue")
    ax.set_title(sku)
    ax.set_ylabel("")
plt.suptitle("Ürün Bazında Satış Adedi Dağılımı (Aykırı Değerler)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "outlier_boxplot.png", dpi=140, bbox_inches="tight")
plt.show()


In [ ]:
q1 = df.groupby("urun_kodu")["satis_adedi"].transform(lambda s: s.quantile(0.25))
q3 = df.groupby("urun_kodu")["satis_adedi"].transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
alt_sinir = (q1 - 1.5 * iqr).clip(lower=0)
ust_sinir = q3 + 1.5 * iqr

n_aykiri = ((df["satis_adedi"] < alt_sinir) | (df["satis_adedi"] > ust_sinir)).sum()
print(f"Toplam aykırı değer sayısı: {n_aykiri} ({n_aykiri/len(df)*100:.2f}%)")

# Ürün bazında özet
ozet = pd.DataFrame({
    "urun_kodu": df["urun_kodu"],
    "aykiri_mi": (df["satis_adedi"] < alt_sinir) | (df["satis_adedi"] > ust_sinir)
})
print(ozet.groupby("urun_kodu")["aykiri_mi"].sum())

**Karar:** Aykırı değerleri **silmek yerine üst/alt sınıra çekiyorum
(winsorize/capping).** Gerekçe:
- Bu bir zaman serisi olduğu için satır silmek, tarih sürekliliğini bozar
  (modelin bir sonraki gün özelliklerini -lag features- hesaplarken boşluk
  oluşturur).
- Çok yüksek değerler muhtemelen gerçek ama nadir "toplu sipariş" günlerini,
  çok düşük değerler ise muhtemelen veri girişi hatalarını temsil ediyor. Her
  iki durumda da ham değer modele **aşırı ağırlıklı ve yanıltıcı bir sinyal**
  verir; sınıra çekmek bu etkiyi dengelerken bilgiyi tamamen yok etmez.
- Capping sonrası orijinal sütunu değil, `satis_adedi` sütununu güncelliyorum
  ki sonraki bölümlerde tek bir "temiz" hedef değişken kullanılsın.

In [ ]:
df["satis_adedi"] = df["satis_adedi"].clip(lower=alt_sinir, upper=ust_sinir)

# Capping etkisini görselleştir
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, (sku, grup) in zip(axes, df.groupby("urun_kodu")):
    sns.boxplot(y=grup["satis_adedi"], ax=ax, color="lightgreen")
    ax.set_title(sku)
    ax.set_ylabel("")
plt.suptitle("Capping Sonrası Ürün Bazında Satış Adedi Dağılımı")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "outlier_boxplot_capped.png", dpi=140, bbox_inches="tight")
plt.show()


### 2.3 Veri Dönüşümü (Encoding) — Ön Hazırlık

`urun_kodu` kategorik bir değişken. Modelleme bölümünde (Bölüm 4) hem
**One-Hot Encoding** hem de ağaç tabanlı modellerin doğrudan işleyebileceği
**Label/Ordinal Encoding** karşılaştırılacak; bu yüzden burada ham kategorik
sütunu koruyoruz, dönüştürme kararını modelleme aşamasında gerekçeleriyle
vereceğiz. Ölçeklendirme (StandardScaler/MinMaxScaler) kararı da Bölüm 4'te,
kullanılacak modele göre (ağaç tabanlı modeller ölçeklendirme gerektirmez,
doğrusal modeller gerektirir) verilecektir.

Temizlenmiş veri setini bir sonraki bölümde (EDA) kullanmak üzere kaydedelim.

In [ ]:
CLEAN_DATA_PATH = WORK_DIR / "talep_verisi_temiz.csv"
df.to_csv(CLEAN_DATA_PATH, index=False, encoding="utf-8-sig")
print("Temizlenmiş veri kaydedildi:", CLEAN_DATA_PATH)
print("Boyut:", df.shape)
df.head()


## Bölüm 3: Keşifsel Veri Analizi (EDA) ve Özellik Mühendisliği

Bu bölümün amacı yalnızca grafik üretmek değil, talebi oluşturan örüntüleri
anlamak ve bu bulguları modelde kullanılabilecek değişkenlere dönüştürmektir.
Zaman serilerinde en kritik konu **veri sızıntısını önlemektir**: Bir günün
tahmininde o günün veya geleceğin gerçek satış değeri özellik olarak
kullanılmayacaktır.


In [ ]:
# Takvim özellikleri
df["yil"] = df["tarih"].dt.year
df["ay"] = df["tarih"].dt.month
df["ayin_gunu"] = df["tarih"].dt.day
df["haftanin_gunu"] = df["tarih"].dt.dayofweek
df["hafta_numarasi"] = df["tarih"].dt.isocalendar().week.astype(int)
df["ceyrek"] = df["tarih"].dt.quarter
df["hafta_sonu_mu"] = (df["haftanin_gunu"] >= 5).astype(int)
df["yaz_mevsimi_mi"] = df["ay"].isin([6, 7, 8]).astype(int)

# Ürünlerin fiyat seviyeleri çok farklı olduğu için ürün içi göreli fiyat
# endeksi oluşturulur. 1.00 ürünün medyan fiyat seviyesini ifade eder.
df["fiyat_endeksi"] = (
    df["birim_fiyat"]
    / df.groupby("urun_kodu")["birim_fiyat"].transform("median")
)

print("Tarih aralığı:", df["tarih"].min().date(), "-", df["tarih"].max().date())
print("Ürün sayısı:", df["urun_kodu"].nunique())
df.head()


### 3.1 Zaman İçindeki Talep ve Ürün Farkları

Aşağıdaki grafik günlük dalgalanmayı okunabilir kılmak için 30 günlük
hareketli ortalamayı gösterir. Her ürünün ayrı eksende çizilmesi, yüksek
hacimli ürünlerin düşük hacimli ürünleri görsel olarak bastırmasını önler.


In [ ]:
gunluk = df[["tarih", "urun_kodu", "satis_adedi"]].copy()
gunluk["otuz_gunluk_ortalama"] = gunluk.groupby("urun_kodu")["satis_adedi"].transform(
    lambda seri: seri.rolling(30, min_periods=7).mean()
)

fig, axes = plt.subplots(5, 1, figsize=(14, 13), sharex=True)
for ax, (sku, grup) in zip(axes, gunluk.groupby("urun_kodu")):
    ax.plot(grup["tarih"], grup["satis_adedi"], color="lightgray", alpha=0.35, linewidth=0.7)
    ax.plot(grup["tarih"], grup["otuz_gunluk_ortalama"], color="#1f77b4", linewidth=1.8)
    ax.set_title(sku, loc="left", fontweight="bold")
    ax.set_ylabel("Adet")
axes[-1].set_xlabel("Tarih")
fig.suptitle("Ürün Bazında Günlük Talep ve 30 Günlük Hareketli Ortalama", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "talep_zaman_serileri.png", dpi=140, bbox_inches="tight")
plt.show()


**Yorum:** Ürünler aynı genel takvim etkilerini taşısa da satış ölçekleri
belirgin biçimde farklıdır. TON80 en yüksek, SOMON ise en düşük hacimli
üründür. Yaz aylarında tekrarlanan yükseliş ve hafta sonlarında kısa süreli
düşüşler görülür. Bu bulgu, tek bir genel ortalama ile tahmin yapılamayacağını;
modele ürün kimliği, takvim değişkenleri ve geçmiş satış gecikmelerinin
eklenmesi gerektiğini gösterir.


### 3.2 Promosyon, Hafta Sonu, Yaz ve Tatil Etkisi

Burada her ikili değişken için 0 ve 1 gruplarının ortalama satışları
karşılaştırılır. Yüzdesel etki, ilgili koşulun bulunduğu günlerin ortalamasının
normal günlere göre farkıdır. Bu analiz nedensellik kanıtlamaz; veri setindeki
ilişkiyi ölçer.


In [ ]:
etki_degisenleri = {
    "Promosyon": "promosyon_var_mi",
    "Hafta sonu": "hafta_sonu_mu",
    "Yaz mevsimi": "yaz_mevsimi_mi",
    "Resmi tatil": "resmi_tatil_mi",
}

etki_satirlari = []
for etiket, sutun in etki_degisenleri.items():
    ortalamalar = df.groupby(sutun)["satis_adedi"].mean()
    normal = ortalamalar.get(0, np.nan)
    kosul = ortalamalar.get(1, np.nan)
    etki_satirlari.append({
        "faktor": etiket,
        "normal_gun_ortalamasi": normal,
        "kosullu_gun_ortalamasi": kosul,
        "yuzdesel_fark": (kosul / normal - 1) * 100,
    })

etki_tablosu = pd.DataFrame(etki_satirlari).round(2)
display(etki_tablosu)

fig, ax = plt.subplots(figsize=(8, 4))
renkler = ["#2ca02c" if deger >= 0 else "#d62728" for deger in etki_tablosu["yuzdesel_fark"]]
sns.barplot(data=etki_tablosu, x="faktor", y="yuzdesel_fark", palette=renkler, hue="faktor", legend=False, ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Normal Günlere Göre Fark (%)")
ax.set_xlabel("")
ax.set_title("Talebi Etkileyen İş ve Takvim Faktörleri")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "is_takvim_etkileri.png", dpi=140, bbox_inches="tight")
plt.show()


**Yorum:** Promosyonlu günlerde ortalama talep yaklaşık **%26 artarken**,
hafta sonlarında yaklaşık **%27 düşmektedir**. Yaz mevsimi yaklaşık **%25
artış**, resmi tatiller ise yaklaşık **%52 düşüş** ile ilişkilidir. Bu nedenle
`promosyon_var_mi`, `hafta_sonu_mu`, `yaz_mevsimi_mi` ve
`resmi_tatil_mi` tahmin anında bilinen ve modele doğrudan eklenmesi gereken
özelliklerdir. Özellikle tatil bilgisinin çıkarılması, modelin tatil günlerinde
sistematik biçimde fazla tahmin üretmesine yol açabilir.


### 3.3 Aylık ve Haftalık Mevsimsellik

Isı haritalarında koyu renkler daha yüksek ortalama talebi gösterir. Birinci
grafik her ürünün aylık desenini, ikinci grafik haftanın günlerine göre
ürün davranışını karşılaştırır.


In [ ]:
aylik_pivot = df.pivot_table(
    index="urun_kodu", columns="ay", values="satis_adedi", aggfunc="mean"
)
haftalik_pivot = df.pivot_table(
    index="urun_kodu", columns="haftanin_gunu", values="satis_adedi", aggfunc="mean"
)
haftalik_pivot.columns = ["Pzt", "Sal", "Çar", "Per", "Cum", "Cmt", "Paz"]

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
sns.heatmap(aylik_pivot, cmap="YlGnBu", annot=True, fmt=".0f", ax=axes[0])
axes[0].set_title("Ürün ve Ay Bazında Ortalama Günlük Talep")
axes[0].set_xlabel("Ay")
axes[0].set_ylabel("Ürün")
sns.heatmap(haftalik_pivot, cmap="YlOrRd", annot=True, fmt=".0f", ax=axes[1])
axes[1].set_title("Ürün ve Haftanın Günü Bazında Ortalama Talep")
axes[1].set_xlabel("Haftanın Günü")
axes[1].set_ylabel("Ürün")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mevsimsellik_isiharitalari.png", dpi=140, bbox_inches="tight")
plt.show()


**Yorum:** Haziran–ağustos döneminde özellikle ton balığı ve sardalya
ürünlerinde talep yükselir. Cumartesi ve pazar günleri ise tüm ürünlerde daha
düşük satış görülür. Bu tekrarlanan yapı nedeniyle ay, haftanın günü, hafta
numarası ve hafta sonu göstergeleri modele eklenmiştir. Yalnızca doğrusal trend
kullanmak bu periyodik hareketleri yakalayamaz.


### 3.4 Korelasyon Analizi

Korelasyon, iki değişkenin birlikte hareketini gösterir; tek başına nedensellik
kanıtı değildir. Ayrıca kategorik ürün etkisi korelasyon tablosunda tam olarak
görülemediğinden modelde ürün kodu ayrı biçimde kodlanacaktır.


In [ ]:
korelasyon_sutunlari = [
    "satis_adedi", "birim_fiyat", "fiyat_endeksi", "promosyon_var_mi",
    "resmi_tatil_mi", "ay", "haftanin_gunu", "hafta_sonu_mu", "yaz_mevsimi_mi"
]
korelasyon = df[korelasyon_sutunlari].corr()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(korelasyon, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Sayısal Değişkenler Korelasyon Matrisi")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "korelasyon_matrisi.png", dpi=140, bbox_inches="tight")
plt.show()

print("Satış ile en yüksek mutlak korelasyonlar:")
display(korelasyon["satis_adedi"].drop("satis_adedi").abs().sort_values(ascending=False))


**Yorum:** `birim_fiyat` ile satış arasında yaklaşık -0,42 korelasyon görülmesi,
tek başına “fiyat artışı talebi bu kadar düşürür” anlamına gelmez. Ürünlerin
temel fiyat ve talep seviyeleri farklıdır; örneğin pahalı bir ürünün doğal satış
hacmi düşük olabilir. Bu nedenle ham fiyat-satış ilişkisi büyük ölçüde **ürün
etkisiyle karışmıştır (confounding)**. Ham fiyatın yanında ürün içi
`fiyat_endeksi` oluşturulmuş ve ürün kodu modele dahil edilmiştir. Böylece
ürünler arası seviye farkı, fiyatın ürün içindeki değişimiyle karıştırılmaz.


### 3.5 Özellik Mühendisliği

Ham veride olmayan aşağıdaki özellikler türetilmiştir:

- **Takvim özellikleri:** yıl, ay, ayın günü, haftanın günü, hafta numarası,
  çeyrek, hafta sonu ve yaz göstergeleri.
- **Ürün içi fiyat endeksi:** Her ürünün fiyatını kendi medyanına göre ifade
  eder; ürünler arası fiyat seviyesi farkını azaltır.
- **Gecikmeli talep (lag):** 1, 7, 14 ve 28 gün önceki satışlar. Özellikle
  `lag_7`, geçen haftanın aynı gününü temsil eder.
- **Hareketli istatistikler:** Önceki 7/28 günün ortalaması ve 7 günlük standart
  sapması; yakın dönem seviye ve oynaklığı yakalar.

Tüm gecikmeli ve hareketli değişkenlerde önce `shift(1)` uygulanmıştır. Böylece
bugünün gerçek satış değeri bugünün tahmininde kullanılamaz ve veri sızıntısı
önlenir.


In [ ]:
urun_grubu = df.groupby("urun_kodu", group_keys=False)["satis_adedi"]

for gecikme in [1, 7, 14, 28]:
    df[f"lag_{gecikme}"] = urun_grubu.shift(gecikme)

for pencere in [7, 28]:
    df[f"hareketli_ortalama_{pencere}"] = urun_grubu.transform(
        lambda seri: seri.shift(1).rolling(pencere).mean()
    )

df["hareketli_std_7"] = urun_grubu.transform(
    lambda seri: seri.shift(1).rolling(7).std()
)

model_df = (
    df.dropna()
      .sort_values(["tarih", "urun_kodu"])
      .reset_index(drop=True)
)

FEATURE_DATA_PATH = WORK_DIR / "talep_verisi_ozellikli.csv"
model_df.to_csv(FEATURE_DATA_PATH, index=False, encoding="utf-8-sig")

print("Özellik mühendisliği öncesi satır sayısı:", len(df))
print("Modellemeye hazır satır sayısı:", len(model_df))
print("İlk 28 günün lag/rolling hesabı nedeniyle kaybedilen satır:", len(df) - len(model_df))
print("Kaydedildi:", FEATURE_DATA_PATH)
model_df.head()


### 3.6 EDA'dan Çıkan En Önemli Üç İçgörü

1. **Promosyon talebi belirgin biçimde yükseltiyor:** Ortalama artış yaklaşık
   %26'dır. Kampanya takvimi üretim planına önceden verilmelidir.
2. **Hafta sonu ve resmi tatil etkisi güçlüdür:** Hafta sonu düşüşü yaklaşık
   %27, resmi tatil düşüşü yaklaşık %52'dir. Takvim bilgisi olmadan model
   sistematik hata yapacaktır.
3. **Ürün ölçeği ve mevsimsellik ürün bazında değişmektedir:** TON80 en yüksek,
   SOMON en düşük hacimli üründür; yaz etkisi bazı ürünlerde daha belirgindir.
   Bu nedenle ürün kodu ve ürün geçmişine ait lag/rolling özellikleri birlikte
   kullanılmalıdır.

Bu bulguların katkısı Bölüm 4'te, yalnızca takvim/iş özelliklerini kullanan
model ile lag/rolling özelliklerini kullanan model karşılaştırılarak ayrıca
test edilecektir.


## Bölüm 4: Modelleme ve Değerlendirme

### 4.1 Metrik Seçimi

Bu çalışma bir **regresyon** problemidir; hedef bir sınıf etiketi değil günlük
satış adedidir. Bu nedenle Accuracy anlamlı değildir.

- **MAE:** Tahminin ortalama kaç adet saptığını doğrudan iş dilinde gösterir.
- **RMSE:** Büyük hataları daha fazla cezalandırır; stoksuz kalma veya fazla
  stok riski yaratan büyük sapmalar için önemlidir.
- **MAPE:** Hatanın yüzde olarak anlaşılmasını sağlar ve <%15 teknik hedefiyle
  doğrudan uyumludur. Çok düşük/sıfır hedeflerde sorunlu olabilir; bu veri
  setinde capping sonrası sıfıra yakın talep bulunmadığı doğrulanmıştır.

Modeller ayrıca geçen haftanın aynı gününü tahmin olarak kullanan
**mevsimsel naive baseline (`lag_7`)** ile karşılaştırılır.


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.inspection import permutation_importance
import joblib


KATEGORIK_OZELLIKLER = ["urun_kodu"]
SAYISAL_OZELLIKLER = [
    "birim_fiyat", "promosyon_var_mi", "resmi_tatil_mi", "yil", "ay",
    "ayin_gunu", "haftanin_gunu", "hafta_numarasi", "ceyrek",
    "hafta_sonu_mu", "yaz_mevsimi_mi", "fiyat_endeksi", "lag_1",
    "lag_7", "lag_14", "lag_28", "hareketli_ortalama_7",
    "hareketli_ortalama_28", "hareketli_std_7",
]
TUM_OZELLIKLER = KATEGORIK_OZELLIKLER + SAYISAL_OZELLIKLER
HEDEF = "satis_adedi"

# Son üç ay nihai test seti olarak tamamen ayrılır.
TEST_BASLANGIC = pd.Timestamp("2024-10-01")
train_df = model_df[model_df["tarih"] < TEST_BASLANGIC].copy().reset_index(drop=True)
test_df = model_df[model_df["tarih"] >= TEST_BASLANGIC].copy().reset_index(drop=True)

X_train, y_train = train_df[TUM_OZELLIKLER], train_df[HEDEF]
X_test, y_test = test_df[TUM_OZELLIKLER], test_df[HEDEF]

print("Eğitim dönemi:", train_df["tarih"].min().date(), "-", train_df["tarih"].max().date())
print("Test dönemi:", test_df["tarih"].min().date(), "-", test_df["tarih"].max().date())
print("Eğitim satırı:", len(train_df), "| Test satırı:", len(test_df))


### 4.2 Dönüşüm ve Zaman Serisi Çapraz Doğrulaması

`urun_kodu` nominaldir; ürünler arasında doğal bir büyüklük sırası yoktur.
Bu yüzden Label Encoding yerine **One-Hot Encoding** kullanılır. Ridge modeli
için sayısal değişkenlere **StandardScaler** uygulanır; çünkü katsayı cezası
ölçekten etkilenir. Ağaç tabanlı modeller eşiklerle bölme yaptığı için
ölçeklendirme gerektirmez.

Rastgele K-Fold kullanılmaz: Gelecekteki gözlemlerin geçmişi tahmin ederken
kullanılması veri sızıntısı yaratır. Bunun yerine tarih bazlı, genişleyen
pencereli **TimeSeriesSplit** kullanılır. Aynı tarihteki beş ürünün farklı
fold'lara ayrılmaması için split önce benzersiz tarihler üzerinde oluşturulur.


In [ ]:
# Benzersiz tarihler üzerinde genişleyen pencere splitleri
egitim_tarihleri = np.array(sorted(train_df["tarih"].unique()))
tscv = TimeSeriesSplit(n_splits=3)
cv_splits = []

for fold, (train_date_idx, valid_date_idx) in enumerate(tscv.split(egitim_tarihleri), start=1):
    fold_train_dates = egitim_tarihleri[train_date_idx]
    fold_valid_dates = egitim_tarihleri[valid_date_idx]
    train_idx = np.flatnonzero(train_df["tarih"].isin(fold_train_dates).to_numpy())
    valid_idx = np.flatnonzero(train_df["tarih"].isin(fold_valid_dates).to_numpy())
    cv_splits.append((train_idx, valid_idx))
    print(
        f"Fold {fold}: eğitim {fold_train_dates[0].date()}–{fold_train_dates[-1].date()} "
        f"({len(train_idx)} satır), doğrulama {fold_valid_dates[0].date()}–"
        f"{fold_valid_dates[-1].date()} ({len(valid_idx)} satır)"
    )

tree_preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), KATEGORIK_OZELLIKLER),
    ("num", "passthrough", SAYISAL_OZELLIKLER),
])

ridge_preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), KATEGORIK_OZELLIKLER),
    ("num", StandardScaler(), SAYISAL_OZELLIKLER),
])

modeller = {
    "Ridge": Pipeline([
        ("preprocess", ridge_preprocess),
        ("model", Ridge(alpha=10.0)),
    ]),
    "Random Forest": Pipeline([
        ("preprocess", tree_preprocess),
        ("model", RandomForestRegressor(
            n_estimators=200, max_depth=16, min_samples_leaf=2,
            max_features=0.9, random_state=42, n_jobs=-1
        )),
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocess", tree_preprocess),
        ("model", GradientBoostingRegressor(
            n_estimators=250, learning_rate=0.05, max_depth=3,
            min_samples_leaf=3, loss="huber", random_state=42
        )),
    ]),
}


In [ ]:
def metrikleri_hesapla(gercek, tahmin):
    return {
        "MAE": mean_absolute_error(gercek, tahmin),
        "RMSE": np.sqrt(mean_squared_error(gercek, tahmin)),
        "MAPE_%": mean_absolute_percentage_error(gercek, tahmin) * 100,
    }


cv_satirlari = []
for model_adi, model in modeller.items():
    fold_metrikleri = []
    for fold, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
        fold_model = clone(model)
        fold_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        fold_tahmin = fold_model.predict(X_train.iloc[valid_idx])
        metrik = metrikleri_hesapla(y_train.iloc[valid_idx], fold_tahmin)
        metrik["fold"] = fold
        fold_metrikleri.append(metrik)

    fold_df = pd.DataFrame(fold_metrikleri)
    cv_satirlari.append({
        "Model": model_adi,
        "CV_MAE_Ortalama": fold_df["MAE"].mean(),
        "CV_MAE_Std": fold_df["MAE"].std(),
        "CV_RMSE_Ortalama": fold_df["RMSE"].mean(),
        "CV_MAPE_Ortalama_%": fold_df["MAPE_%"].mean(),
    })

cv_sonuclari = pd.DataFrame(cv_satirlari).sort_values("CV_MAE_Ortalama").reset_index(drop=True)
display(cv_sonuclari.round(3))
cv_sonuclari.to_csv(OUTPUT_DIR / "cv_model_karsilastirma.csv", index=False, encoding="utf-8-sig")


In [ ]:
# Hiç görmediğimiz son üç aylık test setinde karşılaştırma
test_satirlari = []
test_tahminleri = {}

baseline_tahmin = test_df["lag_7"].to_numpy()
baseline_metrik = metrikleri_hesapla(y_test, baseline_tahmin)
test_satirlari.append({"Model": "Mevsimsel Naive (lag_7)", **baseline_metrik})
test_tahminleri["Mevsimsel Naive (lag_7)"] = baseline_tahmin

for model_adi, model in modeller.items():
    fitted = clone(model).fit(X_train, y_train)
    tahmin = fitted.predict(X_test)
    test_tahminleri[model_adi] = tahmin
    test_satirlari.append({"Model": model_adi, **metrikleri_hesapla(y_test, tahmin)})

test_sonuclari = pd.DataFrame(test_satirlari).sort_values("MAE").reset_index(drop=True)
display(test_sonuclari.round(3))
test_sonuclari.to_csv(OUTPUT_DIR / "test_model_karsilastirma.csv", index=False, encoding="utf-8-sig")


**Model seçimi:** Doğrusal Ridge modeli hızlı ve yorumlanabilir bir temel
modeldir ancak promosyon, tatil ve gecikmeli talep arasındaki doğrusal olmayan
etkileşimleri sınırlı yakalar. Random Forest CV'de en düşük ortalama MAE'yi
vermiştir; Gradient Boosting ile arasındaki yaklaşık 0,85 adetlik fark ise her
iki modelin fold standart sapmasının çok altındadır. Başka bir ifadeyle CV
sonuçları iki ağaç modelini pratikte yakın göstermektedir. Ayrılmış test
döneminde Gradient Boosting daha düşük MAE/RMSE/MAPE üretmiş, ayrıca daha
kompakt ve düzgün tahmin eğrileri sunmuştur. Bu gerekçeler birlikte
belgelenerek Gradient Boosting final aday olarak seçilmiştir; test setine göre
tekrar tekrar model/parametre denenmemiştir.


### 4.3 Özellik Mühendisliğinin Katkı Testi

Lag ve hareketli istatistiklerin gerçekten katkı sağlayıp sağlamadığını görmek
için aynı Gradient Boosting modeli iki kez kurulur. İlk model yalnızca ürün,
fiyat, promosyon ve takvim özelliklerini; ikinci model tüm özellikleri kullanır.
Karşılaştırma aynı test döneminde yapılır.


In [ ]:
TEMEL_OZELLIKLER = [
    "urun_kodu", "birim_fiyat", "promosyon_var_mi", "resmi_tatil_mi",
    "yil", "ay", "ayin_gunu", "haftanin_gunu", "hafta_numarasi", "ceyrek",
    "hafta_sonu_mu", "yaz_mevsimi_mi", "fiyat_endeksi",
]

temel_preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["urun_kodu"]),
    ("num", "passthrough", [x for x in TEMEL_OZELLIKLER if x != "urun_kodu"]),
])
temel_model = Pipeline([
    ("preprocess", temel_preprocess),
    ("model", GradientBoostingRegressor(
        n_estimators=250, learning_rate=0.05, max_depth=3,
        min_samples_leaf=3, loss="huber", random_state=42
    )),
])
temel_model.fit(train_df[TEMEL_OZELLIKLER], y_train)
temel_tahmin = temel_model.predict(test_df[TEMEL_OZELLIKLER])

tam_model = clone(modeller["Gradient Boosting"]).fit(X_train, y_train)
tam_tahmin = tam_model.predict(X_test)

katki_testi = pd.DataFrame([
    {"Özellik Seti": "Yalnızca iş + takvim özellikleri", **metrikleri_hesapla(y_test, temel_tahmin)},
    {"Özellik Seti": "İş + takvim + lag/rolling", **metrikleri_hesapla(y_test, tam_tahmin)},
])
display(katki_testi.round(3))


**Katkı yorumu:** Lag/rolling özellikleri RMSE'yi yaklaşık %2,7 azaltarak büyük
hataları düşürmüş; buna karşılık MAE ve MAPE'de çok küçük bir kötüleşme
yaratmıştır. Bu sonuç “üretilen her özellik her metriği iyileştirir” varsayımının
doğru olmadığını gösterir. İş problemi büyük sapmaların stockout/fazla stok
riski yaratmasına duyarlı olduğu için RMSE iyileşmesi değerlidir; final modelde
lag/rolling özellikleri korunmuş, kısıt bölümünde gerçek veride yeniden
doğrulanmaları gerektiği belirtilmiştir.


### 4.4 Hiperparametre Optimizasyonu

Final aday varsayılan ayarlarla bırakılmamıştır. `RandomizedSearchCV` ile
aşağıdaki alanlar denenir: ağaç sayısı (150–350), öğrenme oranı (0.03–0.08),
maksimum derinlik (2–4), minimum yaprak örneği (3–10), alt örnekleme oranı
(0.8/1.0) ve kayıp fonksiyonu (Huber/kare hata). Arama yalnızca eğitim dönemi
ve aynı TimeSeriesSplit fold'ları üzerinde yapılır; test seti seçim sürecine
katılmaz.


In [ ]:
parametre_dagilimi = {
    "model__n_estimators": [150, 250, 350],
    "model__learning_rate": [0.03, 0.05, 0.08],
    "model__max_depth": [2, 3, 4],
    "model__min_samples_leaf": [3, 5, 10],
    "model__subsample": [0.8, 1.0],
    "model__loss": ["huber", "squared_error"],
}

arama = RandomizedSearchCV(
    estimator=clone(modeller["Gradient Boosting"]),
    param_distributions=parametre_dagilimi,
    n_iter=8,
    scoring="neg_mean_absolute_error",
    cv=cv_splits,
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
    refit=True,
)
arama.fit(X_train, y_train)

print("En iyi parametreler:")
for parametre, deger in arama.best_params_.items():
    print(f"  {parametre}: {deger}")
print(f"En iyi CV MAE: {-arama.best_score_:.3f}")

arama_ozeti = (
    pd.DataFrame(arama.cv_results_)
      .sort_values("rank_test_score")
      [["rank_test_score", "mean_train_score", "mean_test_score", "std_test_score", "params"]]
      .head(5)
)
display(arama_ozeti)


In [ ]:
final_model = arama.best_estimator_
train_tahmin = final_model.predict(X_train)
test_tahmin = final_model.predict(X_test)

final_metrikler = pd.DataFrame([
    {"Veri": "Eğitim", **metrikleri_hesapla(y_train, train_tahmin)},
    {"Veri": "Test", **metrikleri_hesapla(y_test, test_tahmin)},
])
display(final_metrikler.round(3))

urun_metrikleri = []
for sku, indeksler in test_df.groupby("urun_kodu").groups.items():
    pozisyonlar = np.array(list(indeksler))
    urun_metrikleri.append({
        "urun_kodu": sku,
        **metrikleri_hesapla(y_test.iloc[pozisyonlar], test_tahmin[pozisyonlar]),
    })
urun_metrikleri = pd.DataFrame(urun_metrikleri).sort_values("MAPE_%")
display(urun_metrikleri.round(3))

model_yolu = OUTPUT_DIR / "talep_tahmin_modeli.joblib"
joblib.dump(final_model, model_yolu)
print("Final model kaydedildi:", model_yolu)


In [ ]:
# Gerçek ve tahmin karşılaştırması
sonuc_df = test_df[["tarih", "urun_kodu", "urun_adi", "satis_adedi", "lag_7"]].copy()
sonuc_df["model_tahmini"] = test_tahmin
sonuc_df["artik"] = sonuc_df["satis_adedi"] - sonuc_df["model_tahmini"]

fig, axes = plt.subplots(5, 1, figsize=(14, 13), sharex=True)
for ax, (sku, grup) in zip(axes, sonuc_df.groupby("urun_kodu")):
    ax.plot(grup["tarih"], grup["satis_adedi"], label="Gerçek", color="#1f77b4", linewidth=1.5)
    ax.plot(grup["tarih"], grup["model_tahmini"], label="Tahmin", color="#ff7f0e", linewidth=1.4)
    ax.set_title(sku, loc="left", fontweight="bold")
    ax.set_ylabel("Adet")
axes[0].legend(ncol=2, loc="upper right")
axes[-1].set_xlabel("Tarih")
fig.suptitle("Test Döneminde Gerçek ve Tahmin Edilen Günlük Talep", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "gercek_tahmin_karsilastirma.png", dpi=140, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(sonuc_df["artik"], kde=True, ax=axes[0], color="#4c78a8")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Tahmin Hatalarının Dağılımı")
axes[0].set_xlabel("Gerçek - Tahmin")
sns.scatterplot(data=sonuc_df, x="model_tahmini", y="artik", hue="urun_kodu", alpha=0.65, ax=axes[1])
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Tahmin Seviyesine Göre Hata")
axes[1].set_xlabel("Tahmin")
axes[1].set_ylabel("Artık")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "artik_analizi.png", dpi=140, bbox_inches="tight")
plt.show()

sonuc_df.to_csv(OUTPUT_DIR / "test_tahminleri.csv", index=False, encoding="utf-8-sig")


### 4.5 Overfitting Kontrolü

Model başarısı tek bir rastgele train-test ayrımıyla ölçülmemiştir. Üç
genişleyen zaman penceresindeki CV sonuçları modelin farklı dönemlerdeki
istikrarını, son üç aylık dokunulmamış test seti ise gerçek gelecek senaryosunu
ölçer. Eğitim hatasının test hatasından düşük olması beklenir; ancak çok büyük
bir fark ezber işareti olur. `max_depth`, `min_samples_leaf`, `subsample` ve
Huber kaybı model karmaşıklığını/gürültü duyarlılığını sınırlamak için aramaya
dahil edilmiştir. Sonuçlar baseline'dan belirgin biçimde daha iyi ve CV
fold'ları arasında tutarlıysa modelin genelleme yaptığı kabul edilir.


## Bölüm 5: Model Yorumlanabilirliği ve Sonuçlar

### 5.1 Global Özellik Önem Düzeyi

Gradient Boosting'in ağaç tabanlı önem değerleri modelin bölünmelerde hangi
değişkenleri kullandığını gösterir. Buna ek olarak permutation importance,
test setindeki bir özelliği karıştırdığımızda MAE'nin ne kadar bozulduğunu
ölçerek daha doğrudan bir genelleme önemi sunar.


In [ ]:
donusturucu = final_model.named_steps["preprocess"]
agac_modeli = final_model.named_steps["model"]
donusturulmus_adlar = [
    ad.replace("cat__", "").replace("num__", "")
    for ad in donusturucu.get_feature_names_out()
]

agac_onem = (
    pd.DataFrame({"ozellik": donusturulmus_adlar, "onem": agac_modeli.feature_importances_})
      .sort_values("onem", ascending=False)
      .head(15)
)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=agac_onem, x="onem", y="ozellik", color="#4c78a8", ax=ax)
ax.set_title("Gradient Boosting — En Önemli 15 Özellik")
ax.set_xlabel("Ağaç Tabanlı Önem")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=140, bbox_inches="tight")
plt.show()

display(agac_onem.round(4))


In [ ]:
perm = permutation_importance(
    final_model, X_test, y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)
perm_onem = (
    pd.DataFrame({
        "ozellik": TUM_OZELLIKLER,
        "mae_artisi": perm.importances_mean,
        "std": perm.importances_std,
    })
    .sort_values("mae_artisi", ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=perm_onem, x="mae_artisi", y="ozellik", color="#f58518", ax=ax)
ax.set_title("Test Seti Permutation Importance")
ax.set_xlabel("Özellik Karıştırıldığında MAE Artışı")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "permutation_importance.png", dpi=140, bbox_inches="tight")
plt.show()

display(perm_onem.round(4))


**Yorum:** Geçmiş satış gecikmeleri ve hareketli ortalamalar yakın dönem talep
seviyesini taşırken; ürün kodu ürünler arası kalıcı ölçek farkını açıklar.
Promosyon, tatil ve hafta sonu göstergeleri kısa dönemli sıçrama/düşüşleri
düzeltir. Ağaç önemi ile permutation öneminin birlikte değerlendirilmesi,
yalnızca model içi bölünme sayısına dayalı tek bir yoruma bağlı kalmayı önler.


### 5.2 Tek Bir Tahminin Açıklanması

Aşağıda test dönemindeki promosyonlu ve yüksek talep beklenen bir gün seçilir.
Her özellik sırayla aynı ürünün eğitim dönemi medyanına (ikili göstergelerde
normal gün değerine) çekilir. Tahmindeki değişim, o özelliğin bu tek tahmin
üzerindeki yerel etkisini gösterir. Bu yöntem SHAP kadar tam bir katkı
ayrıştırması değildir; ancak yöneticiye “hangi sinyal tahmini ne yönde
değiştirdi?” sorusunu somut bir karşı-olgusal senaryoyla açıklar.


In [ ]:
promosyon_pozisyonlari = np.flatnonzero(X_test["promosyon_var_mi"].to_numpy() == 1)
if len(promosyon_pozisyonlari) > 0:
    yerel_pozisyon = promosyon_pozisyonlari[np.argmax(test_tahmin[promosyon_pozisyonlari])]
else:
    yerel_pozisyon = int(np.argmax(test_tahmin))

yerel_satir = X_test.iloc[[yerel_pozisyon]].copy()
yerel_sku = yerel_satir["urun_kodu"].iloc[0]
temel_tahmin = float(final_model.predict(yerel_satir)[0])
urun_egitim = train_df[train_df["urun_kodu"] == yerel_sku]

ikili_ozellikler = {"promosyon_var_mi", "resmi_tatil_mi", "hafta_sonu_mu", "yaz_mevsimi_mi"}
yerel_etkiler = []
for ozellik in SAYISAL_OZELLIKLER:
    alternatif = yerel_satir.copy()
    referans = 0 if ozellik in ikili_ozellikler else urun_egitim[ozellik].median()
    alternatif[ozellik] = referans
    alternatif_tahmin = float(final_model.predict(alternatif)[0])
    yerel_etkiler.append({
        "ozellik": ozellik,
        "mevcut_deger": float(yerel_satir[ozellik].iloc[0]),
        "referans_deger": float(referans),
        "tahmine_etki_adet": temel_tahmin - alternatif_tahmin,
    })

yerel_etki_df = pd.DataFrame(yerel_etkiler)
yerel_etki_df["mutlak_etki"] = yerel_etki_df["tahmine_etki_adet"].abs()
yerel_etki_df = yerel_etki_df.sort_values("mutlak_etki", ascending=False).head(10)

print("Açıklanan tahmin")
print("Tarih:", test_df.iloc[yerel_pozisyon]["tarih"].date())
print("Ürün:", yerel_sku, "-", test_df.iloc[yerel_pozisyon]["urun_adi"])
print(f"Model tahmini: {temel_tahmin:.1f} adet")
print(f"Gerçek satış: {y_test.iloc[yerel_pozisyon]:.1f} adet")
display(yerel_etki_df.drop(columns="mutlak_etki").round(2))

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = yerel_etki_df.sort_values("tahmine_etki_adet")
renkler = ["#d62728" if x < 0 else "#2ca02c" for x in plot_df["tahmine_etki_adet"]]
ax.barh(plot_df["ozellik"], plot_df["tahmine_etki_adet"], color=renkler)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"Tek Tahmin İçin Yerel Etki — {yerel_sku}")
ax.set_xlabel("Referans Değere Göre Tahmine Etki (Adet)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "yerel_tahmin_aciklamasi.png", dpi=140, bbox_inches="tight")
plt.show()


### 5.3 İsteğe Bağlı SHAP Açıklaması

Kılavuzda SHAP/LIME “mümkünse” istenmiştir. Aşağıdaki hücre `shap` kuruluysa
aynı tek tahmin için waterfall grafiği üretir; kurulu değilse notebook'un ana
akışını durdurmaz. Ana teslim, ek paket olmadan da feature importance,
permutation importance ve yerel karşı-olgusal açıklamayı içerir.


In [ ]:
try:
    import shap

    X_yerel_donusmus = donusturucu.transform(yerel_satir)
    shap_aciklayici = shap.TreeExplainer(agac_modeli)
    shap_degerleri = np.asarray(shap_aciklayici.shap_values(X_yerel_donusmus)).reshape(-1)
    taban_deger = float(np.asarray(shap_aciklayici.expected_value).reshape(-1)[0])

    aciklama = shap.Explanation(
        values=shap_degerleri,
        base_values=taban_deger,
        data=np.asarray(X_yerel_donusmus).reshape(-1),
        feature_names=donusturulmus_adlar,
    )
    shap.plots.waterfall(aciklama, max_display=12, show=False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "shap_tek_tahmin.png", dpi=140, bbox_inches="tight")
    plt.show()
except ImportError:
    print("SHAP kurulu değil. requirements.txt kurulduktan sonra bu hücre SHAP grafiğini üretecektir.")
except Exception as hata:
    print("SHAP isteğe bağlı hücresi çalıştırılamadı:", hata)


### 5.4 Bulguların İş Diline Çevrilmesi

Teknik olmayan yönetici özeti:

> Sistem, her ürün için önümüzdeki dönemin günlük talebini geçmiş satış düzeni,
> kampanya ve çalışma takvimine göre tahmin eder. Böylece planlama ekibi yalnızca
> geçen haftanın satışını kopyalamak yerine yaklaşan promosyon, yaz dönemi,
> hafta sonu ve tatil etkisini hesaba katan bir tahmin kullanır. Öncelik;
> promosyonlu günleri üretim planına önceden aktarmak, yüksek hacimli TON80 ve
> TON160 için emniyet stoğunu ürün bazında hesaplamak ve tatil günlerinde fazla
> üretimi önlemektir.

Aşağıda yedi günlük tedarik/üretim süresi ve %95 hizmet seviyesi varsayımıyla
örnek emniyet stoğu ve yeniden sipariş noktası hesaplanır. Bunlar gerçek stok,
tedarik süresi ve maliyet verileri geldiğinde güncellenmesi gereken karar
destek değerleridir.


In [ ]:
baseline_metrikleri = metrikleri_hesapla(y_test, baseline_tahmin)
model_metrikleri = metrikleri_hesapla(y_test, test_tahmin)
rmse_iyilesme = (baseline_metrikleri["RMSE"] - model_metrikleri["RMSE"]) / baseline_metrikleri["RMSE"] * 100

# Gerçeğin tahmini %15'ten fazla aşması, basit bir stoksuz kalma risk göstergesidir.
baseline_risk = int((y_test.to_numpy() > baseline_tahmin * 1.15).sum())
model_risk = int((y_test.to_numpy() > test_tahmin * 1.15).sum())

print(f"Baseline'a göre RMSE iyileşmesi: %{rmse_iyilesme:.1f}")
print(f"%15'ten büyük eksik tahmin günü — baseline: {baseline_risk}, model: {model_risk}")
print("Not: Bu risk sayısı gerçek stockout değil, stok verisi olmadığı için kullanılan tahmin temelli bir proxy'dir.")

TEDARIK_SURESI_GUN = 7
Z_95 = 1.65
stok_onerileri = []
for sku, grup in sonuc_df.groupby("urun_kodu"):
    hata_std = grup["artik"].std()
    ortalama_gunluk_tahmin = grup["model_tahmini"].mean()
    emniyet_stogu = Z_95 * hata_std * np.sqrt(TEDARIK_SURESI_GUN)
    yeniden_siparis = ortalama_gunluk_tahmin * TEDARIK_SURESI_GUN + emniyet_stogu
    stok_onerileri.append({
        "urun_kodu": sku,
        "ortalama_gunluk_tahmin": ortalama_gunluk_tahmin,
        "emniyet_stogu_adet": emniyet_stogu,
        "yeniden_siparis_noktasi_adet": yeniden_siparis,
    })

stok_onerileri = pd.DataFrame(stok_onerileri).round(0)
display(stok_onerileri)
stok_onerileri.to_csv(OUTPUT_DIR / "stok_onerileri.csv", index=False, encoding="utf-8-sig")


### 5.5 Başarı Kriterlerinin Sonucu

- Final modelin test **MAPE değeri %8,26** olup <%15 hedefini karşılamıştır.
- Test **RMSE değeri 52,40**, mevsimsel baseline'ın 94,34 değerine göre yaklaşık
  **%44,5 iyileşme** göstermiştir.
- %15'ten büyük eksik tahmini risk göstergesi baseline'da 62 günden modelde 40
  güne düşmüştür (**yaklaşık %35,5 azalma**). Bu gerçek stockout sayısı değil,
  stok verisi bulunmadığı için açıkça “proxy” olarak raporlanmıştır.
- Eğitim ve test MAE değerleri sırasıyla 35,05 ve 35,15'tir. Aradaki çok küçük
  fark, final modelde belirgin bir overfitting işareti olmadığını destekler.

Teknik hedefler karşılanmıştır. Gerçek iş metriği olan stockout ve fazla stok
maliyetinin doğrulanması için gerçek stok hareketleriyle pilot uygulama
gereklidir.


## Bölüm 6: Projenin Kısıtları ve Gelecek Çalışmalar

### 6.1 Zayıf Yönler ve Kısıtlar

1. **Sentetik veri:** Veri, kontrollü bir senaryodan üretilmiştir; gerçek müşteri,
   kanal, bölge ve rakip davranışlarını tam olarak temsil etmez. Sonuçlar gerçek
   Dardanel performansı olarak yorumlanamaz.
2. **Satış talebe eşit varsayılmıştır:** Gerçek dünyada stok bittiğinde kayıtlı
   satış gerçek talebi olduğundan düşük gösterir. Stokout/censored demand bilgisi
   veri setinde yoktur.
3. **Dış değişkenler eksiktir:** Hava durumu, rakip fiyatı, enflasyon, döviz,
   mağaza/kanal ve pazarlama harcaması kullanılmamıştır.
4. **Üç yıllık dönem:** Uzun dönemli yapısal kırılmaları ve daha geniş ekonomik
   döngüleri öğrenmek için sınırlıdır.
5. **Promosyon etkisi nedensel değildir:** Model promosyon ile satış arasındaki
   ilişkiyi öğrenir; kampanyanın gerçek nedensel artışını deneysel olarak ölçmez.
6. **Envanter önerisi basitleştirilmiştir:** Sabit yedi günlük tedarik süresi ve
   %95 hizmet seviyesi varsayılmış; elde bulundurma, sipariş ve fire maliyetleri
   optimize edilmemiştir.

### 6.2 Bir Ay Daha Olsaydı Yapılacaklar

- Oracle ERP'den anonimleştirilmiş gerçek sipariş, stok, üretim ve stockout
  kayıtlarını alıp modeli gerçek veriyle yeniden eğitmek.
- Hava durumu, kampanya bütçesi, kanal/bölge ve rakip fiyatı gibi dış kaynakları
  eklemek.
- LightGBM/XGBoost ve olasılıksal tahmin modellerini karşılaştırıp yalnızca
  nokta tahmini değil %80/%95 tahmin aralığı üretmek.
- Ürün hiyerarşisi ve yeni ürün problemi için hiyerarşik tahmin yaklaşımı
  denemek.
- Tahminleri gerçek stok politikasıyla bir backtest/simülasyonda test ederek
  stockout, fazla stok ve toplam maliyet değişimini ölçmek.
- Modeli FastAPI ile servis edip veri kayması ve tahmin hatası için izleme
  paneli kurmak; aylık yeniden eğitim kuralı tanımlamak.

### Sonuç

Proje; iş problemini tanımlamadan veri temizleme, EDA, özellik mühendisliği,
zaman uyumlu çapraz doğrulama, model karşılaştırma, hiperparametre optimizasyonu,
yorumlanabilirlik ve iş aksiyonlarına kadar uçtan uca tekrarlanabilir bir talep
tahmini süreci sunmaktadır. Nihai doğruluk tek başına amaç değildir; tahminin
üretim ve stok kararına nasıl dönüştürüleceği, hangi varsayımlara dayandığı ve
hangi koşullarda yeniden doğrulanması gerektiği açıkça belirtilmiştir.


## Tekrarlanabilirlik Kontrol Listesi

- Ham veri: `talep_verisi_ham.csv`
- Sentetik veri üretim kodu: `generate_data.py`
- Tüm analiz ve modelleme: bu notebook (`proje.ipynb`)
- Bağımlılıklar: `requirements.txt`
- Üretilen grafik, tablo, tahmin ve model dosyaları: `outputs/`

Notebook baştan sona çalıştırıldığında temiz/özellikli CSV'leri ve tüm çıktıları
ham verinin bulunduğu klasöre otomatik olarak üretir. Rastgelelik içeren bütün
adımlarda `random_state=42` kullanılmıştır.
